# 🦷 DentalGemma → GGUF Conversion (Multimodal)

**Converts the finetuned DentalGemma 1.5 4B IT model to GGUF format
with multimodal (vision) support for on-device mobile deployment.**

### What this notebook does:
1. Installs llama.cpp and builds conversion/quantization tools
2. Downloads `naazimsnh02/dentalgemma-1.5-4b-it` from HuggingFace
3. Converts safetensors → GGUF (text model)
4. Extracts SigLIP vision encoder → mmproj GGUF
5. Quantizes to Q4_K_M (~2.5 GB) for mobile
6. Verifies with a test inference
7. Packages both files for download

**Runtime:** Use a **GPU runtime** (T4 is fine — GPU is only needed for
optional test inference, conversion itself is CPU-bound).

**Time:** ~15-25 minutes total.

## 1. Install Dependencies

In [1]:
import os
import sys

if "google.colab" in sys.modules and not os.environ.get("VERTEX_PRODUCT"):
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
else:
    if os.environ.get("VERTEX_PRODUCT") == "COLAB_ENTERPRISE":
        os.environ["HF_HOME"] = "/content/hf"
    from huggingface_hub import get_token
    if get_token() is None:
        from huggingface_hub import notebook_login
        notebook_login()

In [2]:
import os
from huggingface_hub import get_token

# 1. Check if the env var is injected
if "HF_TOKEN" in os.environ:
    print("✅ Success! 'HF_TOKEN' environment variable is present.")
    print(f"Token starts with: {os.environ['HF_TOKEN'][:4]}****")
else:
    print("❌ 'HF_TOKEN' environment variable is MISSING.")

# 2. Check what Hugging Face sees
token = get_token()
if token:
    print(f"✅ Hugging Face Login Status: Authenticated (Token found)")
else:
    print("❌ Hugging Face Login Status: Not Logged In")

✅ Success! 'HF_TOKEN' environment variable is present.
Token starts with: hf_u****
✅ Hugging Face Login Status: Authenticated (Token found)


In [3]:
# Uninstall incompatible PyTorch and torchvision versions
!pip uninstall torch torchvision torchaudio

Found existing installation: torch 2.10.0+cpu
Uninstalling torch-2.10.0+cpu:
  Would remove:
    /usr/local/bin/torchfrtrace
    /usr/local/bin/torchrun
    /usr/local/lib/python3.12/dist-packages/functorch/*
    /usr/local/lib/python3.12/dist-packages/torch-2.10.0+cpu.dist-info/*
    /usr/local/lib/python3.12/dist-packages/torch/*
    /usr/local/lib/python3.12/dist-packages/torchgen/*
Proceed (Y/n)? y
y
y
y
  Successfully uninstalled torch-2.10.0+cpu
Found existing installation: torchvision 0.25.0+cpu
Uninstalling torchvision-0.25.0+cpu:
  Would remove:
    /usr/local/lib/python3.12/dist-packages/torchvision-0.25.0+cpu.dist-info/*
    /usr/local/lib/python3.12/dist-packages/torchvision.libs/libjpeg.4af9affd.so.8
    /usr/local/lib/python3.12/dist-packages/torchvision.libs/libpng16.c2edc9e1.so.16
    /usr/local/lib/python3.12/dist-packages/torchvision.libs/libsharpyuv.994c9d2c.so.0
    /usr/local/lib/python3.12/dist-packages/torchvision.libs/libwebp.87a45f40.so.7
    /usr/local/lib/pyt

In [4]:
# Install compatible PyTorch and torchvision versions
!pip install torch==2.9.0+cu128 torchvision==0.24.0+cu128 --index-url https://download.pytorch.org/whl/cu128

Looking in indexes: https://download.pytorch.org/whl/cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 76.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 218.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 154.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 53.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 63.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 69.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 126.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 78.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 73.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 MB 51.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 83.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.7/

In [5]:
# Install other dependencies
!pip install -q huggingface_hub hf_transfer accelerate bitsandbytes datasets evaluate peft tensorboard transformers trl sentencepiece protobuf gguf numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.5/540.5 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.2/96.2 kB 4.2 MB/s eta 0:00:00


In [6]:
# Enable fast HuggingFace downloads
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

## 2. Clone & Build llama.cpp

We need three tools from llama.cpp:
- `convert_hf_to_gguf.py` — converts safetensors to GGUF
- `gemma3_convert_encoder_to_gguf.py` — extracts the SigLIP vision encoder
- `llama-quantize` — quantizes the GGUF to smaller sizes
- `llama-mtmd-cli` — (optional) test multimodal inference

In [7]:
# Clone llama.cpp (latest stable)
!git clone --depth 1 https://github.com/ggml-org/llama.cpp.git /content/llama.cpp

Cloning into '/content/llama.cpp'...
remote: Enumerating objects: 2549, done.
remote: Counting objects: 100% (2549/2549), done.
remote: Compressing objects: 100% (2039/2039), done.
remote: Total 2549 (delta 514), reused 1649 (delta 439), pack-reused 0 (from 0)
Receiving objects: 100% (2549/2549), 27.54 MiB | 15.13 MiB/s, done.
Resolving deltas: 100% (514/514), done.
Updating files: 100% (2278/2278), done.


In [8]:
# Determine the correct multimodal CLI target
# Modern llama.cpp uses `llama-mtmd-cli`, older experimental branches used `llama-gemma3-cli`
import os

cmake_path = "/content/llama.cpp/CMakeLists.txt"
target_cli = "llama-mtmd-cli" # Default for Feb 2026

if os.path.exists(cmake_path):
    with open(cmake_path, "r") as f:
        content = f.read()
        if "llama-mtmd-cli" in content:
            target_cli = "llama-mtmd-cli"
        elif "llama-gemma3-cli" in content:
            target_cli = "llama-gemma3-cli"
        elif "llama-llava-cli" in content:
            # Fallback for generic multimodal if specific ones aren't found
            target_cli = "llama-llava-cli"

print(f"🎯 Targeted Multimodal CLI: {target_cli}")

🎯 Targeted Multimodal CLI: llama-mtmd-cli


In [9]:
# Build llama.cpp with the detected target
!cd /content/llama.cpp && cmake -B build \
    -DBUILD_SHARED_LIBS=OFF \
    -DGGML_CUDA=OFF \
    -DLLAMA_CURL=OFF && \
    cmake --build build -j$(nproc) --target llama-quantize {target_cli} llama-cli

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- Found OpenMP_C: 

In [10]:
# Verify builds
!ls -lh /content/llama.cpp/build/bin/llama-quantize
!ls -lh /content/llama.cpp/build/bin/{target_cli}

-rwxr-xr-x 1 root root 4.6M Feb 20 16:27 /content/llama.cpp/build/bin/llama-quantize
-rwxr-xr-x 1 root root 9.3M Feb 20 16:28 /content/llama.cpp/build/bin/llama-mtmd-cli


## 3. Download DentalGemma from HuggingFace

Downloads the full finetuned model (~9 GB).
If you have a private model, set your HF token first:
```python
from huggingface_hub import login
login(token="hf_YOUR_TOKEN")
```

In [11]:
from huggingface_hub import snapshot_download

MODEL_ID = "naazimsnh02/dentalgemma-1.5-4b-it"
MODEL_DIR = "/content/dentalgemma-model"
OUTPUT_DIR = "/content/dentalgemma-gguf"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"📥 Downloading {MODEL_ID}...")
snapshot_download(
    repo_id=MODEL_ID,
    local_dir=MODEL_DIR,
    ignore_patterns=["*.md", ".gitattributes"],
)
print("✅ Download complete!")

📥 Downloading naazimsnh02/dentalgemma-1.5-4b-it...


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

✅ Download complete!


In [12]:
# Verify downloaded files
!ls -lh {MODEL_DIR}/

total 9.3G
-rw-r--r-- 1 root root 1.5K Feb 20 16:31 chat_template.jinja
-rw-r--r-- 1 root root 2.7K Feb 20 16:31 config.json
-rw-r--r-- 1 root root  114 Feb 20 16:31 generation_config.json
-rw-r--r-- 1 root root 9.3G Feb 20 16:33 model.safetensors
-rw-r--r-- 1 root root  580 Feb 20 16:31 processor_config.json
-rw-r--r-- 1 root root  743 Feb 20 16:31 tokenizer_config.json
-rw-r--r-- 1 root root  32M Feb 20 16:31 tokenizer.json


## 4. Convert Safetensors → GGUF (Text Model)

This converts the full model weights to GGUF format in BF16 precision.
The BF16 GGUF will be ~7.8 GB — we'll quantize it next.

**Patch:** MedGemma 1.5 (Jan 2026) ships a tokenizer whose hash isn't
registered in llama.cpp yet. We patch `convert_hf_to_gguf.py` to add it.

In [13]:
# Patch llama.cpp's converter to recognize DentalGemma/MedGemma 1.5 tokenizer
CONVERT_SCRIPT = "/content/llama.cpp/convert_hf_to_gguf.py"
TOKENIZER_HASH = "789696f5946cc0fc59371f39f6097cafed196b3acded6140432f26bbb1ae1669"

with open(CONVERT_SCRIPT, "r") as f:
    lines = f.readlines()

if TOKENIZER_HASH not in "".join(lines):
    # Find the line with the raise NotImplementedError about BPE pre-tokenizer
    # and insert our hash check just before it, matching the file's indentation
    new_lines = []
    patched = False
    for line in lines:
        if not patched and 'raise NotImplementedError("BPE pre-tokenizer was not recognized' in line:
            # Detect the indentation of the raise line
            indent = line[:len(line) - len(line.lstrip())]
            # Insert our hash check before the raise, turning the raise into an else
            new_lines.append(f'{indent}if chkhsh == "{TOKENIZER_HASH}":\n')
            new_lines.append(f'{indent}    # DentalGemma / MedGemma 1.5 (Gemma 3 tokenizer)\n')
            new_lines.append(f'{indent}    res = "default"\n')
            new_lines.append(f'{indent}else:\n')
            new_lines.append(f'{indent}    {line.lstrip()}')
            patched = True
        else:
            new_lines.append(line)

    if patched:
        with open(CONVERT_SCRIPT, "w") as f:
            f.writelines(new_lines)
        print(f"✅ Patched convert_hf_to_gguf.py with tokenizer hash: {TOKENIZER_HASH[:16]}...")
    else:
        print("⚠️ Could not find the raise NotImplementedError line to patch")
else:
    print("✅ Tokenizer hash already recognized — no patch needed")

✅ Patched convert_hf_to_gguf.py with tokenizer hash: 789696f5946cc0fc...


In [14]:
!cd /content/llama.cpp && python convert_hf_to_gguf.py \
    {MODEL_DIR} \
    --outfile {OUTPUT_DIR}/dentalgemma-4b-bf16.gguf \
    --outtype bf16

INFO:hf-to-gguf:Loading model: dentalgemma-model
INFO:hf-to-gguf:Model architecture: Gemma3ForConditionalGeneration
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,                     torch.bfloat16 --> BF16, shape = {2560, 262208}
INFO:numexpr.utils:NumExpr defaulting to 2 threads.


INFO:hf-to-gguf:'▁▁' is encoded and decoded back to '  ' using AutoTokenizer
INFO:hf-to-gguf:'▁▁▁' is encoded and decoded back to '   ' using AutoTokenizer
INFO:hf-to-gguf:'▁▁▁▁' is encoded and decoded back to '    ' using AutoTokenizer
INFO:hf-to-gguf:'▁▁▁▁▁' is encoded and decoded back to '     ' using AutoTokenizer
INFO:hf-to-gguf:'▁▁▁▁▁▁' is encoded and decoded back to '      ' using AutoTokenizer
INFO:hf-to-gguf:'▁▁▁▁▁▁▁' is encoded and decoded back to '       ' using AutoTokenizer
INFO:hf-to-gguf:'▁▁▁▁▁▁▁▁' is encoded and decoded back to '        ' usin

In [15]:
!ls -lh {OUTPUT_DIR}/dentalgemma-4b-bf16.gguf

-rw-r--r-- 1 root root 8.5G Feb 20 16:37 /content/dentalgemma-gguf/dentalgemma-4b-bf16.gguf


## 5. Extract Vision Encoder → mmproj GGUF

The SigLIP vision encoder is needed for image understanding.
This creates a separate `mmproj.gguf` file (~860 MB).

In [18]:
# Install any extra deps the vision conversion script might need
!pip install -q pillow

In [19]:
# Extract the mmproj (vision encoder + projector) using convert_hf_to_gguf.py
# The --mmproj flag tells it to export only the vision/projector tensors
!cd /content/llama.cpp && python convert_hf_to_gguf.py \
    {MODEL_DIR} \
    --outfile {OUTPUT_DIR}/dentalgemma-mmproj-bf16.gguf \
    --outtype bf16 \
    --mmproj

INFO:hf-to-gguf:Loading model: dentalgemma-model
INFO:hf-to-gguf:Model architecture: Gemma3ForConditionalGeneration
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:mm.input_projection.weight,      torch.bfloat16 --> F16, shape = {2560, 1152}
INFO:hf-to-gguf:Correcting norm value for 'multi_modal_projector.mm_soft_emb_norm.weight'
INFO:hf-to-gguf:mm.soft_emb_norm.weight,         torch.bfloat16 --> F32, shape = {1152}
INFO:hf-to-gguf:v.patch_embd.bias,               torch.bfloat16 --> F32, shape = {1152}
INFO:hf-to-gguf:v.patch_embd.weight,             torch.bfloat16 --> F32, shape = {14, 14, 3, 1152}
INFO:hf-to-gguf:v.position_embd.weight,          torch.bfloat16 --> F32, shape = {1152, 4096}
INFO:hf-to-gguf:v.blk.0.ln1.bias,                torch.bfloat16 --> F32, shape = {1152}
INFO:hf-to-gguf:v.blk.0.ln1.weight,              torch.bfloat16 --> F32, shap

In [20]:
!ls -lh {OUTPUT_DIR}/dentalgemma-mmproj-bf16.gguf

total 812M
-rw-r--r-- 1 root root 812M Feb 20 16:50 mmproj-Dentalgemma-Model-BF16.gguf


In [21]:
!ls -lh {OUTPUT_DIR}/dentalgemma-mmproj-bf16.gguf

total 812M
-rw-r--r-- 1 root root 812M Feb 20 16:50 mmproj-Dentalgemma-Model-BF16.gguf


## 6. Quantize Text Model

Quantize from BF16 (~7.8 GB) to smaller formats for mobile deployment.
We create two variants:
- **Q4_K_M** (~2.5 GB) — Best balance of quality vs size for flagships

In [22]:
# Q4_K_M — Recommended for most devices (8+ GB RAM)
!/content/llama.cpp/build/bin/llama-quantize \
    {OUTPUT_DIR}/dentalgemma-4b-bf16.gguf \
    {OUTPUT_DIR}/dentalgemma-4b-Q4_K_M.gguf \
    Q4_K_M

main: build = 1 (b908baf)
main: built with GNU 11.4.0 for Linux x86_64
main: quantizing '/content/dentalgemma-gguf/dentalgemma-4b-bf16.gguf' to '/content/dentalgemma-gguf/dentalgemma-4b-Q4_K_M.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 33 key-value pairs and 445 tensors from /content/dentalgemma-gguf/dentalgemma-4b-bf16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = gemma3
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Dentalgemma Model
llama_model_loader: - kv   3:                         general.size_label str              = 4.6B
llama_model_loader: - kv   4:                         gemma3.block_count u32              = 34
llama_model_loader: - kv   5:     

In [31]:
# Summary of all generated files
print("=" * 60)
print("📦 Generated GGUF Files:")
print("=" * 60)
!ls -lh {OUTPUT_DIR}/*.gguf
print()
print("For mobile deployment, you need TWO files:")
print("  1. Text model:   dentalgemma-4b-Q4_K_M.gguf")
print("  2. Vision model: dentalgemma-mmproj-f16.gguf (or bf16)")
print()
print(f"Total size (Q4_K_M + mmproj): ", end="")
!du -ch {OUTPUT_DIR}/dentalgemma-4b-Q4_K_M.gguf {OUTPUT_DIR}/dentalgemma-mmproj-bf16.gguf/mmproj-Dentalgemma-Model-BF16.gguf | tail -1

📦 Generated GGUF Files:
-rw-r--r-- 1 root root 8.5G Feb 20 16:37 /content/dentalgemma-gguf/dentalgemma-4b-bf16.gguf
-rw-r--r-- 1 root root 2.7G Feb 20 17:06 /content/dentalgemma-gguf/dentalgemma-4b-Q4_K_M.gguf

/content/dentalgemma-gguf/dentalgemma-mmproj-bf16.gguf:
total 812M
-rw-r--r-- 1 root root 812M Feb 20 16:50 mmproj-Dentalgemma-Model-BF16.gguf

For mobile deployment, you need TWO files:
  1. Text model:   dentalgemma-4b-Q4_K_M.gguf
  2. Vision model: dentalgemma-mmproj-f16.gguf (or bf16)

Total size (Q4_K_M + mmproj): 3.5G	total


## 7. Test Inference (Optional)

Quick sanity check that the converted model works.
Requires GPU runtime for reasonable speed.

### 7a. Text-only test

In [ ]:
!/content/llama.cpp/build/bin/llama-cli \
    -m {OUTPUT_DIR}/dentalgemma-4b-Q4_K_M.gguf \
    -p "<start_of_turn>user\nA 35-year-old patient presents with severe throbbing pain in the lower right molar. What are the possible diagnoses?<end_of_turn>\n<start_of_turn>model\n" \
    --no-conversation \
    -n 200 \
    --temp 0.7 \
    --top-k 40 \
    --top-p 0.95 \
    -ngl 99

### 7b. Multimodal test (image + text)

Download a sample dental X-ray and test vision inference.

In [ ]:
# Upload a dental X-ray image for testing
from google.colab import files

print("📤 Upload a dental X-ray image (JPG/PNG)...")
uploaded = files.upload()

if uploaded:
    test_image_name = list(uploaded.keys())[0]
    import shutil
    shutil.move(test_image_name, "/content/test_xray.jpg")
    print(f"✅ Uploaded '{test_image_name}' → /content/test_xray.jpg")
else:
    print("⚠️ No image uploaded — skipping vision test")

In [ ]:
# Test multimodal inference (text + image)
# This uses the dynamically determined multimodal CLI (llama-mtmd-cli or equivalent)
if os.path.exists("/content/test_xray.jpg"):
    print("🔍 Running multimodal inference test...")
    !echo "Analyze this dental X-ray for any abnormalities." | \
        /content/llama.cpp/build/bin/{target_cli} \
        -m {OUTPUT_DIR}/dentalgemma-4b-Q4_K_M.gguf \
        --mmproj {OUTPUT_DIR}/dentalgemma-mmproj-f16.gguf \
        --image /content/test_xray.jpg \
        -n 200 \
        -ngl 99 \
        --no-conversation \
        -p "<start_of_turn>user\nAnalyze this dental X-ray for any abnormalities.<end_of_turn>\n<start_of_turn>model\n"
else:
    print("⚠️ No test image available — skipping multimodal test")

## 8. Download Converted Files

Download the two files you need for your mobile app:
1. **Text model** (Q4_K_M) — the main LLM
2. **Vision model** (mmproj) — the SigLIP image encoder

You can also optionally delete the large BF16 file to free up space.

In [ ]:
# (Optional) Remove the large BF16 file to free disk space
# Uncomment the line below if you're running low on storage
!rm {OUTPUT_DIR}/dentalgemma-4b-bf16.gguf

In [39]:
# Zip the files for easier download
!cd {OUTPUT_DIR} && zip -j /content/dentalgemma-gguf-Q4_K_M.zip \
    dentalgemma-4b-Q4_K_M.gguf \
    dentalgemma-mmproj-bf16.gguf/mmproj-Dentalgemma-Model-BF16.gguf

!ls -lh /content/dentalgemma-gguf-Q4_K_M.zip

  adding: dentalgemma-4b-Q4_K_M.gguf (deflated 2%)
  adding: mmproj-Dentalgemma-Model-BF16.gguf (deflated 21%)
-rw-r--r-- 1 root root 3.3G Feb 20 17:33 /content/dentalgemma-gguf-Q4_K_M.zip


In [40]:
# Download via Colab (only works in Colab)
try:
    from google.colab import files
    print("📥 Starting download... (this may take a few minutes for ~3 GB)")
    files.download("/content/dentalgemma-gguf-Q4_K_M.zip")
except ImportError:
    print("Not running in Colab — download manually from:")
    print(f"  /content/dentalgemma-gguf-Q4_K_M.zip")

📥 Starting download... (this may take a few minutes for ~3 GB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Alternative: Upload to HuggingFace Hub

If the file is too large to download directly, push it to a HF repo instead.

In [37]:
# Uncomment and run this cell to upload to HuggingFace
from huggingface_hub import HfApi, login

api = HfApi()
REPO_ID = "naazimsnh02/dentalgemma-1.5-4b-it-GGUF"

# # Create the repo if it doesn't exist
api.create_repo(repo_id=REPO_ID, exist_ok=True)

# # Upload Q4_K_M
api.upload_file(
     path_or_fileobj=f"{OUTPUT_DIR}/dentalgemma-4b-Q4_K_M.gguf",
     path_in_repo="dentalgemma-4b-Q4_K_M.gguf",
     repo_id=REPO_ID,
 )
# # Upload mmproj
api.upload_file(
     path_or_fileobj=f"{OUTPUT_DIR}/dentalgemma-mmproj-bf16.gguf/mmproj-Dentalgemma-Model-BF16.gguf",
     path_in_repo="dentalgemma-mmproj-f16.gguf",
     repo_id=REPO_ID,
 )
print(f"✅ Uploaded to https://huggingface.co/{REPO_ID}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...entalgemma-4b-Q4_K_M.gguf:   0%|          |  585kB / 2.88GB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ntalgemma-Model-BF16.gguf:   1%|          | 4.85MB /  851MB            

✅ Uploaded to https://huggingface.co/naazimsnh02/dentalgemma-1.5-4b-it-GGUF


## ✅ Done!

You now have two GGUF files ready for mobile deployment:

| File | Size | Purpose |
|------|------|---------|
| `dentalgemma-4b-Q4_K_M.gguf` | ~2.5 GB | Main text/chat model |
| `dentalgemma-mmproj-f16.gguf` | ~860 MB | SigLIP vision encoder for X-ray images |

**Total: ~3.4 GB** — fits on modern smartphones (8+ GB RAM).

### Troubleshooting

If vision encoder conversion fails:
1. Check llama.cpp version: `cd /content/llama.cpp && git log -1 --oneline`
2. List available conversion scripts: `find /content/llama.cpp -name "*convert*.py"`
3. Try the Unsloth alternative above (uncomment the code)
4. Use pre-converted GGUF models from HuggingFace (search for "GGUF" repos)

### Next Steps
Use these files in your React Native mobile app with `llama.rn` or
a custom native bridge to llama.cpp.